In [1]:
%pip install -q gymnasium==0.29.1 ale-py==0.9.1 stable-baselines3==2.3.2 "autorom[accept-rom-license]" opencv-python
!AutoROM --accept-license > /dev/null 2>&1
print("✅ Install done")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 434.7/434.7 kB 10.6 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 953.9/953.9 kB 48.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 93.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 182.3/182.3 kB 17.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dopamine-rl 4.1.2 requires ale-py>=0.10.1, but you have ale-py 0.9.1 which is incompatible.
dopamine-rl 4.1.2 requires gymnasium>=1.0.0, but you have gymnasium 0.29.1 which is incompatible.
✅ Install done


In [2]:
!git clone https://github.com/Muen1/dqn-pong-formative3.git
%cd dqn-pong-formative3
print("✅ Repo cloned")

Cloning into 'dqn-pong-formative3'...
remote: Enumerating objects: 144, done.
remote: Counting objects: 100% (15/15), done.
remote: Compressing objects: 100% (12/12), done.
remote: Total 144 (delta 5), reused 8 (delta 3), pack-reused 129 (from 4)
Receiving objects: 100% (144/144), 347.19 MiB | 22.84 MiB/s, done.
Resolving deltas: 100% (34/34), done.
Updating files: 100% (61/61), done.
/content/dqn-pong-formative3
✅ Repo cloned


In [3]:
!sed -i 's/member_c_/exp_c_/g' src/evaluate_exp_c.py
!grep -n "exp_c_01\|OUTPUT = " src/evaluate_exp_c.py
print("✅ Script fixed — you should see exp_c_ names above")

16:OUTPUT = Path("experiments/exp_c_results.csv")
19:    ("exp_c_01_baseline", 0.0001, 0.99, 32, 1.00, 0.05, 0.10),
✅ Script fixed — you should see exp_c_ names above


In [4]:
!python src/exp_c_train.py --run_name exp_c_05_low_start_epsilon --eps_start 0.5 --eps_end 0.05 --eps_decay_frac 0.1 --timesteps 50000
print("✅ Retraining done")

2026-07-18 23:34:55.670379: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
A.L.E: Arcade Learning Environment (version 0.9.1+aff5939)
[Powered by Stella]
Using cuda device
Wrapping the env in a VecTransposeImage.
Logging to ./tb_logs/exp_c_05_low_start_epsilon_1
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 3.44e+03 |
|    ep_rew_mean      | 

In [5]:
import subprocess
for name in ["exp_c_02_fast_decay", "exp_c_05_low_start_epsilon"]:
    subprocess.run(["unzip","-oq",f"models/{name}.zip","policy.pth","-d",f"/tmp/{name}"])
h2 = subprocess.check_output(["md5sum",f"/tmp/exp_c_02_fast_decay/policy.pth"]).split()[0]
h5 = subprocess.check_output(["md5sum",f"/tmp/exp_c_05_low_start_epsilon/policy.pth"]).split()[0]
print("exp_02 weights:", h2.decode())
print("exp_05 weights:", h5.decode())
print("✅ Different — fix worked!" if h2 != h5 else "❌ Still identical — re-run Cell 4 and check for errors")

exp_02 weights: 7648811c13f809eaf7934e5757b0f541
exp_05 weights: a95c01f93ec21feb561791e52e2208be
✅ Different — fix worked!


In [6]:
!python src/evaluate_exp_c.py
print("✅ Evaluation done")

2026-07-18 23:38:10.672946: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.

Evaluating exp_c_01_baseline...
A.L.E: Arcade Learning Environment (version 0.9.1+aff5939)
[Powered by Stella]
/usr/local/lib/python3.12/dist-packages/stable_baselines3/common/save_util.py:167: UserWarning: Could not deserialize object _last_obs. Consider using `custom_objects` argument to replace this object.
Exce

In [7]:
import pandas as pd
df = pd.read_csv("experiments/exp_c_results.csv")
display(df[["experiment","run_name","evaluation_mean_reward","evaluation_reward_std","evaluation_mean_length","best_training_reward"]])
lengths = df["evaluation_mean_length"].nunique()
print(f"\nUnique episode lengths across 10 rows: {lengths}")
print("✅ Evaluation is real now!" if lengths > 1 else "❌ Still identical — something is wrong, share this output with Claude")

,experiment,run_name,evaluation_mean_reward,evaluation_reward_std,evaluation_mean_length,best_training_reward
0,C07,exp_c_07_low_lr_high_gamma,-20.6,0.49,5556.4,-16.0
1,C01,exp_c_01_baseline,-21.0,0.00,3056.0,-18.0
2,C02,exp_c_02_fast_decay,-21.0,0.00,3056.0,-18.0
3,C03,exp_c_03_slow_decay,-21.0,0.00,3056.0,-18.0
4,C04,exp_c_04_high_final_epsilon,-21.0,0.00,3056.0,-18.0
5,C05,exp_c_05_low_start_epsilon,-21.0,0.00,3056.0,-18.0
6,C06,exp_c_06_high_lr_low_gamma,-21.0,0.00,3056.0,-18.0
7,C08,exp_c_08_aggressive_edge,-21.0,0.00,3056.0,-18.0
8,C09,exp_c_09_large_batch,-21.0,0.00,3056.0,-18.0
9,C10,exp_c_10_small_batch,-21.0,0.00,3056.0,-18.0



Unique episode lengths across 10 rows: 2
✅ Evaluation is real now!


In [8]:
!zip -j fixed_files.zip models/exp_c_05_low_start_epsilon.zip logs/exp_c_05_low_start_epsilon_episodes.csv experiments/exp_c_results.csv src/evaluate_exp_c.py
from google.colab import files
files.download("fixed_files.zip")

  adding: exp_c_05_low_start_epsilon.zip (stored 0%)
  adding: exp_c_05_low_start_epsilon_episodes.csv (deflated 74%)
  adding: exp_c_results.csv (deflated 66%)
  adding: evaluate_exp_c.py (deflated 66%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [9]:
!sed -i 's/deterministic=True/deterministic=False/' src/evaluate_exp_c.py
!python src/evaluate_exp_c.py
import pandas as pd
df = pd.read_csv("experiments/exp_c_results.csv")
display(df[["experiment","evaluation_mean_reward","evaluation_reward_std","evaluation_mean_length"]])
from google.colab import files
files.download("experiments/exp_c_results.csv")

2026-07-18 23:47:30.151111: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.

Evaluating exp_c_01_baseline...
A.L.E: Arcade Learning Environment (version 0.9.1+aff5939)
[Powered by Stella]
/usr/local/lib/python3.12/dist-packages/stable_baselines3/common/save_util.py:167: UserWarning: Could not deserialize object _last_obs. Consider using `custom_objects` argument to replace this object.
Exce

,experiment,evaluation_mean_reward,evaluation_reward_std,evaluation_mean_length
0,C04,-20.4,0.49,3340.2
1,C07,-20.4,0.80,5727.0
2,C10,-20.6,0.49,3196.6
3,C05,-20.8,0.40,3149.4
4,C06,-20.8,0.40,3223.2
5,C01,-21.0,0.00,3152.0
6,C02,-21.0,0.00,3056.0
7,C03,-21.0,0.00,3056.0
8,C08,-21.0,0.00,3056.0
9,C09,-21.0,0.00,3056.0


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>